In [1]:
import sys,os
sys.path.append(r'C:/data/EnergyTrading/Python/')

# from datetime import datetime, time
import pandas as pd
from Database.TPData import TPData
from Database.DB_reader import Database
from datetime import date, timedelta

import psycopg2
import pandas as pd
from sqlalchemy import create_engine

In [2]:
postgre_table_name = 'public.trayport_vw_trades'
# Table details in timescaledb
table_name = "trades"
time_column = "datetime"

# Parameters
batch_size = 10_000  # You can tune this depending on your memory/DB speed

In [3]:
# 1. Read data from source DB
conn = Database()

query=f"""select * from  {postgre_table_name} limit 100""" # where rownum <= 100"""
#query="""select * from  public.trayport_orders limit 100"""

df=conn.execute(query)


print(f"✅ Loaded {len(df)} rows from source database.")

# 2. Write data to target DB (create table if not exists)
conn = Database('timescaledb')
conn._connect()

# Write to new table in TimescaleDB
df.to_sql(table_name, conn.engine, index=False, if_exists='replace')  # 'append' if already exists

print(f"✅ Data written to target TimescaleDB.")


conn.execute_general_query("CREATE EXTENSION IF NOT EXISTS timescaledb;")

# Convert table to hypertable, allow migrating existing rows
conn.execute_general_query(f"""
    SELECT create_hypertable('{table_name}', '{time_column}',
                             if_not_exists => TRUE,
                             migrate_data => TRUE);
""")
print(f"✅ Table '{table_name}' converted to hypertable (with data migration).")

Connected to the database postgre
Disconnected from the database postgre
✅ Loaded 100 rows from source database.
Connected to the database timescaledb
✅ Data written to target TimescaleDB.
Connected to the database timescaledb
Query Error: This result object does not return rows. It has been closed automatically.
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Table 'trades' converted to hypertable (with data migration).


In [4]:
conn = Database('timescaledb')

conn.execute_general_query(f'delete from {table_name}')

Connected to the database timescaledb
Query Error: This result object does not return rows. It has been closed automatically.
Disconnected from the database timescaledb


In [5]:
# 1. Read data from source DB
conn = Database()
query = f"""SELECT * FROM {postgre_table_name}"""
df = conn.execute(query)

print(f"✅ Loaded {len(df)} rows from source database.")

# 2. Connect to TimescaleDB
conn = Database('timescaledb')
conn._connect()

# 3. Insert in batches
total_rows = len(df)
for start in range(0, total_rows, batch_size):
    end = min(start + batch_size, total_rows)
    batch = df.iloc[start:end]
    
    batch.to_sql(table_name, conn.engine, index=False, if_exists='append')
    print(f"✅ Inserted rows {start} to {end} into TimescaleDB.")

print("🎉 All batches inserted successfully.")

Connected to the database postgre
Disconnected from the database postgre
✅ Loaded 16015747 rows from source database.
Connected to the database timescaledb
✅ Inserted rows 0 to 10000 into TimescaleDB.
✅ Inserted rows 10000 to 20000 into TimescaleDB.
✅ Inserted rows 20000 to 30000 into TimescaleDB.
✅ Inserted rows 30000 to 40000 into TimescaleDB.
✅ Inserted rows 40000 to 50000 into TimescaleDB.
✅ Inserted rows 50000 to 60000 into TimescaleDB.
✅ Inserted rows 60000 to 70000 into TimescaleDB.
✅ Inserted rows 70000 to 80000 into TimescaleDB.
✅ Inserted rows 80000 to 90000 into TimescaleDB.
✅ Inserted rows 90000 to 100000 into TimescaleDB.
✅ Inserted rows 100000 to 110000 into TimescaleDB.
✅ Inserted rows 110000 to 120000 into TimescaleDB.
✅ Inserted rows 120000 to 130000 into TimescaleDB.
✅ Inserted rows 130000 to 140000 into TimescaleDB.
✅ Inserted rows 140000 to 150000 into TimescaleDB.
✅ Inserted rows 150000 to 160000 into TimescaleDB.
✅ Inserted rows 160000 to 170000 into TimescaleDB.


✅ Inserted rows 1560000 to 1570000 into TimescaleDB.
✅ Inserted rows 1570000 to 1580000 into TimescaleDB.
✅ Inserted rows 1580000 to 1590000 into TimescaleDB.
✅ Inserted rows 1590000 to 1600000 into TimescaleDB.
✅ Inserted rows 1600000 to 1610000 into TimescaleDB.
✅ Inserted rows 1610000 to 1620000 into TimescaleDB.
✅ Inserted rows 1620000 to 1630000 into TimescaleDB.
✅ Inserted rows 1630000 to 1640000 into TimescaleDB.
✅ Inserted rows 1640000 to 1650000 into TimescaleDB.
✅ Inserted rows 1650000 to 1660000 into TimescaleDB.
✅ Inserted rows 1660000 to 1670000 into TimescaleDB.
✅ Inserted rows 1670000 to 1680000 into TimescaleDB.
✅ Inserted rows 1680000 to 1690000 into TimescaleDB.
✅ Inserted rows 1690000 to 1700000 into TimescaleDB.
✅ Inserted rows 1700000 to 1710000 into TimescaleDB.
✅ Inserted rows 1710000 to 1720000 into TimescaleDB.
✅ Inserted rows 1720000 to 1730000 into TimescaleDB.
✅ Inserted rows 1730000 to 1740000 into TimescaleDB.
✅ Inserted rows 1740000 to 1750000 into Timesc

✅ Inserted rows 3110000 to 3120000 into TimescaleDB.
✅ Inserted rows 3120000 to 3130000 into TimescaleDB.
✅ Inserted rows 3130000 to 3140000 into TimescaleDB.
✅ Inserted rows 3140000 to 3150000 into TimescaleDB.
✅ Inserted rows 3150000 to 3160000 into TimescaleDB.
✅ Inserted rows 3160000 to 3170000 into TimescaleDB.
✅ Inserted rows 3170000 to 3180000 into TimescaleDB.
✅ Inserted rows 3180000 to 3190000 into TimescaleDB.
✅ Inserted rows 3190000 to 3200000 into TimescaleDB.
✅ Inserted rows 3200000 to 3210000 into TimescaleDB.
✅ Inserted rows 3210000 to 3220000 into TimescaleDB.
✅ Inserted rows 3220000 to 3230000 into TimescaleDB.
✅ Inserted rows 3230000 to 3240000 into TimescaleDB.
✅ Inserted rows 3240000 to 3250000 into TimescaleDB.
✅ Inserted rows 3250000 to 3260000 into TimescaleDB.
✅ Inserted rows 3260000 to 3270000 into TimescaleDB.
✅ Inserted rows 3270000 to 3280000 into TimescaleDB.
✅ Inserted rows 3280000 to 3290000 into TimescaleDB.
✅ Inserted rows 3290000 to 3300000 into Timesc

✅ Inserted rows 4660000 to 4670000 into TimescaleDB.
✅ Inserted rows 4670000 to 4680000 into TimescaleDB.
✅ Inserted rows 4680000 to 4690000 into TimescaleDB.
✅ Inserted rows 4690000 to 4700000 into TimescaleDB.
✅ Inserted rows 4700000 to 4710000 into TimescaleDB.
✅ Inserted rows 4710000 to 4720000 into TimescaleDB.
✅ Inserted rows 4720000 to 4730000 into TimescaleDB.
✅ Inserted rows 4730000 to 4740000 into TimescaleDB.
✅ Inserted rows 4740000 to 4750000 into TimescaleDB.
✅ Inserted rows 4750000 to 4760000 into TimescaleDB.
✅ Inserted rows 4760000 to 4770000 into TimescaleDB.
✅ Inserted rows 4770000 to 4780000 into TimescaleDB.
✅ Inserted rows 4780000 to 4790000 into TimescaleDB.
✅ Inserted rows 4790000 to 4800000 into TimescaleDB.
✅ Inserted rows 4800000 to 4810000 into TimescaleDB.
✅ Inserted rows 4810000 to 4820000 into TimescaleDB.
✅ Inserted rows 4820000 to 4830000 into TimescaleDB.
✅ Inserted rows 4830000 to 4840000 into TimescaleDB.
✅ Inserted rows 4840000 to 4850000 into Timesc

✅ Inserted rows 6210000 to 6220000 into TimescaleDB.
✅ Inserted rows 6220000 to 6230000 into TimescaleDB.
✅ Inserted rows 6230000 to 6240000 into TimescaleDB.
✅ Inserted rows 6240000 to 6250000 into TimescaleDB.
✅ Inserted rows 6250000 to 6260000 into TimescaleDB.
✅ Inserted rows 6260000 to 6270000 into TimescaleDB.
✅ Inserted rows 6270000 to 6280000 into TimescaleDB.
✅ Inserted rows 6280000 to 6290000 into TimescaleDB.
✅ Inserted rows 6290000 to 6300000 into TimescaleDB.
✅ Inserted rows 6300000 to 6310000 into TimescaleDB.
✅ Inserted rows 6310000 to 6320000 into TimescaleDB.
✅ Inserted rows 6320000 to 6330000 into TimescaleDB.
✅ Inserted rows 6330000 to 6340000 into TimescaleDB.
✅ Inserted rows 6340000 to 6350000 into TimescaleDB.
✅ Inserted rows 6350000 to 6360000 into TimescaleDB.
✅ Inserted rows 6360000 to 6370000 into TimescaleDB.
✅ Inserted rows 6370000 to 6380000 into TimescaleDB.
✅ Inserted rows 6380000 to 6390000 into TimescaleDB.
✅ Inserted rows 6390000 to 6400000 into Timesc

...

✅ Inserted rows 9310000 to 9320000 into TimescaleDB.
✅ Inserted rows 9320000 to 9330000 into TimescaleDB.
✅ Inserted rows 9330000 to 9340000 into TimescaleDB.
✅ Inserted rows 9340000 to 9350000 into TimescaleDB.
✅ Inserted rows 9350000 to 9360000 into TimescaleDB.
✅ Inserted rows 9360000 to 9370000 into TimescaleDB.
✅ Inserted rows 9370000 to 9380000 into TimescaleDB.
✅ Inserted rows 9380000 to 9390000 into TimescaleDB.
✅ Inserted rows 9390000 to 9400000 into TimescaleDB.
✅ Inserted rows 9400000 to 9410000 into TimescaleDB.
✅ Inserted rows 9410000 to 9420000 into TimescaleDB.
✅ Inserted rows 9420000 to 9430000 into TimescaleDB.
✅ Inserted rows 9430000 to 9440000 into TimescaleDB.
✅ Inserted rows 9440000 to 9450000 into TimescaleDB.
✅ Inserted rows 9450000 to 9460000 into TimescaleDB.
✅ Inserted rows 9460000 to 9470000 into TimescaleDB.
✅ Inserted rows 9470000 to 9480000 into TimescaleDB.
✅ Inserted rows 9480000 to 9490000 into TimescaleDB.
✅ Inserted rows 9490000 to 9500000 into Timesc

✅ Inserted rows 10830000 to 10840000 into TimescaleDB.
✅ Inserted rows 10840000 to 10850000 into TimescaleDB.
✅ Inserted rows 10850000 to 10860000 into TimescaleDB.
✅ Inserted rows 10860000 to 10870000 into TimescaleDB.
✅ Inserted rows 10870000 to 10880000 into TimescaleDB.
✅ Inserted rows 10880000 to 10890000 into TimescaleDB.
✅ Inserted rows 10890000 to 10900000 into TimescaleDB.
✅ Inserted rows 10900000 to 10910000 into TimescaleDB.
✅ Inserted rows 10910000 to 10920000 into TimescaleDB.
✅ Inserted rows 10920000 to 10930000 into TimescaleDB.
✅ Inserted rows 10930000 to 10940000 into TimescaleDB.
✅ Inserted rows 10940000 to 10950000 into TimescaleDB.
✅ Inserted rows 10950000 to 10960000 into TimescaleDB.
✅ Inserted rows 10960000 to 10970000 into TimescaleDB.
✅ Inserted rows 10970000 to 10980000 into TimescaleDB.
✅ Inserted rows 10980000 to 10990000 into TimescaleDB.
✅ Inserted rows 10990000 to 11000000 into TimescaleDB.
✅ Inserted rows 11000000 to 11010000 into TimescaleDB.
✅ Inserted

✅ Inserted rows 12320000 to 12330000 into TimescaleDB.
✅ Inserted rows 12330000 to 12340000 into TimescaleDB.
✅ Inserted rows 12340000 to 12350000 into TimescaleDB.
✅ Inserted rows 12350000 to 12360000 into TimescaleDB.
✅ Inserted rows 12360000 to 12370000 into TimescaleDB.
✅ Inserted rows 12370000 to 12380000 into TimescaleDB.
✅ Inserted rows 12380000 to 12390000 into TimescaleDB.
✅ Inserted rows 12390000 to 12400000 into TimescaleDB.
✅ Inserted rows 12400000 to 12410000 into TimescaleDB.
✅ Inserted rows 12410000 to 12420000 into TimescaleDB.
✅ Inserted rows 12420000 to 12430000 into TimescaleDB.
✅ Inserted rows 12430000 to 12440000 into TimescaleDB.
✅ Inserted rows 12440000 to 12450000 into TimescaleDB.
✅ Inserted rows 12450000 to 12460000 into TimescaleDB.
✅ Inserted rows 12460000 to 12470000 into TimescaleDB.
✅ Inserted rows 12470000 to 12480000 into TimescaleDB.
✅ Inserted rows 12480000 to 12490000 into TimescaleDB.
✅ Inserted rows 12490000 to 12500000 into TimescaleDB.
✅ Inserted

✅ Inserted rows 13810000 to 13820000 into TimescaleDB.
✅ Inserted rows 13820000 to 13830000 into TimescaleDB.
✅ Inserted rows 13830000 to 13840000 into TimescaleDB.
✅ Inserted rows 13840000 to 13850000 into TimescaleDB.
✅ Inserted rows 13850000 to 13860000 into TimescaleDB.
✅ Inserted rows 13860000 to 13870000 into TimescaleDB.
✅ Inserted rows 13870000 to 13880000 into TimescaleDB.
✅ Inserted rows 13880000 to 13890000 into TimescaleDB.
✅ Inserted rows 13890000 to 13900000 into TimescaleDB.
✅ Inserted rows 13900000 to 13910000 into TimescaleDB.
✅ Inserted rows 13910000 to 13920000 into TimescaleDB.
✅ Inserted rows 13920000 to 13930000 into TimescaleDB.
✅ Inserted rows 13930000 to 13940000 into TimescaleDB.
✅ Inserted rows 13940000 to 13950000 into TimescaleDB.
✅ Inserted rows 13950000 to 13960000 into TimescaleDB.
✅ Inserted rows 13960000 to 13970000 into TimescaleDB.
✅ Inserted rows 13970000 to 13980000 into TimescaleDB.
✅ Inserted rows 13980000 to 13990000 into TimescaleDB.
✅ Inserted

✅ Inserted rows 15300000 to 15310000 into TimescaleDB.
✅ Inserted rows 15310000 to 15320000 into TimescaleDB.
✅ Inserted rows 15320000 to 15330000 into TimescaleDB.
✅ Inserted rows 15330000 to 15340000 into TimescaleDB.
✅ Inserted rows 15340000 to 15350000 into TimescaleDB.
✅ Inserted rows 15350000 to 15360000 into TimescaleDB.
✅ Inserted rows 15360000 to 15370000 into TimescaleDB.
✅ Inserted rows 15370000 to 15380000 into TimescaleDB.
✅ Inserted rows 15380000 to 15390000 into TimescaleDB.
✅ Inserted rows 15390000 to 15400000 into TimescaleDB.
✅ Inserted rows 15400000 to 15410000 into TimescaleDB.
✅ Inserted rows 15410000 to 15420000 into TimescaleDB.
✅ Inserted rows 15420000 to 15430000 into TimescaleDB.
✅ Inserted rows 15430000 to 15440000 into TimescaleDB.
✅ Inserted rows 15440000 to 15450000 into TimescaleDB.
✅ Inserted rows 15450000 to 15460000 into TimescaleDB.
✅ Inserted rows 15460000 to 15470000 into TimescaleDB.
✅ Inserted rows 15470000 to 15480000 into TimescaleDB.
✅ Inserted